<!-- track-identity-card -->
# Within-type clustering and cross-type agreement

| | |
|---|---|
| Pipeline step | `05_corner_type_clustering.ipynb` |
| Manuscript section | 4.2 |
| Copied from | `notebooks/NB7A7b_corner_type_viz_v2_2026-07-19.ipynb` |
| Source sha256 | `ab856da68f91fb12ee220b1219d8a5dc` |

**Reads**

- `data/fingerprints/corner_type_profiles_<policy>/corner_type_pivot.parquet`
- `.../corner_type_profile_crosstrack.parquet`

**Writes**

- `data/fingerprints/corner_type_profiles_<policy>/nb7a7b_info.json`

Produces the pooled and within-type silhouettes, the cluster sizes and the adjusted Rand index pairs reported in Section 4.2. KMeans is run with random_state=42 and n_init=20 on min-max scaled features.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


# NB7A.7b — Corner-Type Radar Chart + Cluster Analizi

**Amac:** FAZ 7A.7 corner-type profiling ciktisini gorsellestirir ve corner-type bazli kumeleme yapar.

**Giris verileri:** `data/fingerprints/corner_type_profiles/` (4 parquet)
- `corner_type_pivot` (26x21): 7 metrik x 3 corner type, driver_id index
- `corner_type_profile_crosstrack` (78x11): 26 driver x 3 corner type
- `corner_type_metrics` (183x11): track-level detay
- `corner_type_delta` (26x5): ozet delta metrikleri

**Gorsellistirme kararlari:**
- Birincil: seaborn clustermap (Engle 2017 BMC Bioinformatics)
- Destekleyici: radar chart (Duan 2023 polygon area bias nedeniyle demote)
- Okabe-Ito CVD-safe palette (Wong 2011)
- Redundant coding: renk + marker (Wilke dataviz)

## 1. Imports ve Stil Ayarlari

In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
%pip install plotly

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  KOSU AYARI — SADECE BURAYI DEGISTIR             ║
# ╚══════════════════════════════════════════════════╝
POLICY = "P_EXC"   # "P_INC" = A-blogu DAHIL | "P_EXC" = A-blogu HARIC

# --- ORTAK KIMLIK POLITIKASI (NB11 v3 ile AYNI liste) ---
EXCLUDE_HARD = [
    "20240213_000000_ACAI",   # oyun-ici yapay surucu
    "20240308_ensemble",      # RL politika ciktisi
    "20240501_MPC",           # klasik kontrol baseline (veri kumesi belgesi)
]
ABLOCK = [
    "20240410_A_12_123",
    "20240410_A_21_231",
    "20240411_A_12_312",
]
assert POLICY in ("P_INC", "P_EXC"), "POLICY 'P_INC' ya da 'P_EXC' olmali"
EXCLUDE_IDS = EXCLUDE_HARD + ([] if POLICY == "P_INC" else ABLOCK)
POL_SUF = "_inc" if POLICY == "P_INC" else "_exc"

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from pathlib import Path
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

# scienceplots
try:
    import scienceplots
    plt.style.use(['science', 'ieee', 'no-latex', 'grid'])
    print('scienceplots aktif')
except ImportError:
    plt.style.use('seaborn-v0_8-whitegrid')
    print('scienceplots yok — pip install SciencePlots')

# === RENK PALETELERI ===
OKABE_ITO = ['#000000', '#E69F00', '#56B4E9', '#009E73',
             '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

CLUSTER_COLORS = {0: '#E69F00', 1: '#56B4E9', 2: '#009E73'}
CLUSTER_NAMES  = {0: 'Cluster A', 1: 'Cluster B', 2: 'Cluster C'}
CLUSTER_MARKERS = {0: 'o', 1: 's', 2: '^'}

CORNER_COLORS  = {'slow': '#D55E00', 'medium': '#E69F00', 'fast': '#56B4E9'}
CORNER_MARKERS = {'slow': 'o', 'medium': 's', 'fast': '^'}
CORNER_ORDER   = ['slow', 'medium', 'fast']

# === RCPARAMS ===
mpl.rcParams.update({
    'font.family': 'serif',
    'font.size': 9, 'axes.labelsize': 9, 'axes.titlesize': 10,
    'legend.fontsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.8, 'lines.linewidth': 1.2,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
})

# === YOLLAR ===
THESIS_ROOT = TRACK_ROOT
# v2: NB7A7'nin AYNI politikayla yazdigi dizini okur
CORNER_DIR  = THESIS_ROOT / 'data' / 'fingerprints' / ('corner_type_profiles' + POL_SUF)
FIG_DIR     = THESIS_ROOT / 'results' / 'figures' / ('corner_type' + POL_SUF)
assert CORNER_DIR.exists(), f'{CORNER_DIR} yok — once NB7A7 v2 ayni POLICY ile calistirilmali'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{name}.png', dpi=600, transparent=True, bbox_inches='tight')
    print(f'Kaydedildi: {name}.pdf/.png')

# Kisa driver ID (son 4+ karakter)
def short_id(did):
    parts = did.split('_')
    return parts[-1] if len(parts) >= 3 else did[:8]

print('Kurulum tamam.')

## 2. Veri Yukleme

In [ ]:
# Yukle
df_delta = pd.read_parquet(CORNER_DIR / 'corner_type_delta.parquet')
df_metrics = pd.read_parquet(CORNER_DIR / 'corner_type_metrics.parquet')
df_pivot = pd.read_parquet(CORNER_DIR / 'corner_type_pivot.parquet')
df_cross = pd.read_parquet(CORNER_DIR / 'corner_type_profile_crosstrack.parquet')

print(f'delta:     {df_delta.shape}')
print(f'metrics:   {df_metrics.shape}')
print(f'pivot:     {df_pivot.shape}')
print(f'crosstrack:{df_cross.shape}')

# driver_id kisa isimleri olustur
all_drivers = df_delta['driver_id'].unique()
driver_short = {d: short_id(d) for d in all_drivers}
print(f'\n{len(all_drivers)} surucu: {list(driver_short.values())}')

## 3. Degenerate Surucu Tespiti

ACAI sabit 51.66 km/h, 0 braking, 0 trail — muhtemelen SAC veya hatali oturum.

In [ ]:
# Degenerate saptama: braking_dist ve trail_pct hep 0 olan suruculer
# v2 NOT: kimlik-tabanli dislama artik NB7A7 v2'de YUKARIDA yapiliyor.
# Bu davranissal maske ikinci guvenlik agi olarak KORUNDU; normalde 0 bulmali.
degenerate_mask = (
    (df_pivot.filter(like='mean_braking_dist').sum(axis=1) == 0) &
    (df_pivot.filter(like='trail_pct').sum(axis=1) == 0)
)

degenerate_drivers = df_pivot.index[degenerate_mask].tolist() if hasattr(df_pivot.index, 'name') and df_pivot.index.name == 'driver_id' else []

# Eger index driver_id degilse, delta'dan kontrol et
if not degenerate_drivers:
    degen_from_delta = df_delta[
        (df_delta['braking_delta'] == 0) & 
        (df_delta['trail_slow'] == 0) & 
        (df_delta['trail_fast'] == 0) &
        (df_delta['apex_speed_ratio'] >= 0.99)
    ]['driver_id'].tolist()
    degenerate_drivers = degen_from_delta

print(f'Kimlik politikasi (NB7A7 v2 ile ayni olmali): {POLICY}')
print(f'Davranissal maske — degenerate ({len(degenerate_drivers)}): {degenerate_drivers or "yok (beklenen)"}')
print()

# Pivot df'yi filtrele — analizde kullanilacak temiz versiyon
if df_pivot.index.name == 'driver_id':
    df_pivot_clean = df_pivot.drop(index=degenerate_drivers, errors='ignore')
else:
    df_pivot_clean = df_pivot.copy()  # index driver_id degilse ayri handle

df_cross_clean = df_cross[~df_cross['driver_id'].isin(degenerate_drivers)].copy()

n_clean = len(df_pivot_clean)
print(f'Temiz surucu sayisi: {n_clean} (toplam {len(df_pivot)} - {len(degenerate_drivers)} degenerate)')
print(f'Crosstrack temiz: {len(df_cross_clean)} satir ({n_clean} surucu x 3 corner type)')

## 4. Clustermap — Surucu x Metrik Isisi Haritasi

Birincil gorsellistirme. Z-score per column, Ward linkage, vlag colormap.
21 kolon: 7 metrik x 3 corner type (slow/medium/fast).

**Referans:** Engle et al. 2017, BMC Bioinformatics — gapmaps.

In [ ]:
# --- CLUSTERMAP ---
# Pivot'u z-score normalize et
data_for_heatmap = df_pivot_clean.copy()

# Kisa driver isimleri
if data_for_heatmap.index.name == 'driver_id':
    data_for_heatmap.index = [short_id(d) for d in data_for_heatmap.index]

# Kolon isimlerini kisalt
col_rename = {}
for col in data_for_heatmap.columns:
    parts = col.split('_', 1)  # slow_mean_apex_speed -> slow, mean_apex_speed
    corner = parts[0][0].upper()  # S, M, F
    metric = parts[1].replace('mean_', '').replace('std_', 'sd_')
    col_rename[col] = f'{corner}_{metric}'
data_for_heatmap = data_for_heatmap.rename(columns=col_rename)

# Corner-type kolon renkleri
col_color_map = {'S': CORNER_COLORS['slow'], 'M': CORNER_COLORS['medium'], 'F': CORNER_COLORS['fast']}
col_colors = [col_color_map[c[0]] for c in data_for_heatmap.columns]

# Clustermap
g = sns.clustermap(
    data_for_heatmap,
    z_score=1,              # normalize per column
    method='ward',
    metric='euclidean',
    cmap='vlag',
    center=0,
    col_colors=col_colors,
    figsize=(12, 8),
    dendrogram_ratio=(0.15, 0.12),
    cbar_pos=(0.02, 0.82, 0.03, 0.15),
    linewidths=0.3,
    xticklabels=True,
    yticklabels=True,
)

g.ax_heatmap.set_xlabel('Metrik (S=Slow, M=Medium, F=Fast)', fontsize=9)
g.ax_heatmap.set_ylabel('Surucu', fontsize=9)
g.fig.suptitle('Corner-Type Surucu Profili (Z-Score, Ward Linkage)', y=1.02, fontsize=11)

# Corner-type legend
legend_patches = [Patch(facecolor=CORNER_COLORS[ct], label=ct.capitalize()) for ct in CORNER_ORDER]
g.ax_heatmap.legend(handles=legend_patches, title='Corner Type', 
                     loc='upper left', bbox_to_anchor=(1.15, 1.0), fontsize=8)

save_fig(g.fig, 'clustermap_corner_type_21metrics')
plt.show()

## 5. K-Means Kumeleme (k=3, Pivot Data)

21-boyutlu pivot uzerinde k=3 kumeleme. Validity indices + cluster profilleri.

In [ ]:
# --- K-MEANS k=3 ---
# Min-Max normalize (radar chart uyumlu, onceki kararla tutarli)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df_pivot_clean)

# K-Means
km = KMeans(n_clusters=3, random_state=42, n_init=20)
labels_km = km.fit_predict(X_scaled)

# Validity indices
sil = silhouette_score(X_scaled, labels_km)
ch  = calinski_harabasz_score(X_scaled, labels_km)
db  = davies_bouldin_score(X_scaled, labels_km)

print(f'=== K-Means k=3 (21 boyut, {len(X_scaled)} surucu) ===')
print(f'Silhouette: {sil:.3f}')
print(f'Calinski-Harabasz: {ch:.2f}')
print(f'Davies-Bouldin: {db:.3f}')
print()

# Cluster atamalari
if df_pivot_clean.index.name == 'driver_id':
    driver_ids = df_pivot_clean.index.tolist()
else:
    driver_ids = list(range(len(df_pivot_clean)))

cluster_df = pd.DataFrame({
    'driver_id': driver_ids,
    'cluster': labels_km,
    'driver_short': [short_id(d) if isinstance(d, str) else str(d) for d in driver_ids]
})

# Cluster boyutlari
for c in sorted(cluster_df['cluster'].unique()):
    members = cluster_df[cluster_df['cluster'] == c]['driver_short'].tolist()
    print(f'Cluster {c} ({len(members)}): {members}')

## 6. Cluster Centroid Profilleri — Grouped Bar Chart

Her corner-type x metrik icin cluster centroid'lerini karsilastirir.
Clustermap'in numerik destegi.

In [ ]:
# --- CLUSTER CENTROID GROUPED BAR ---
# Her cluster icin ortalama profil
df_pivot_labeled = df_pivot_clean.copy()
df_pivot_labeled['cluster'] = labels_km

# Min-Max scaled centroidler (0-1 arasi, karsilastirmaya uygun)
df_scaled = pd.DataFrame(X_scaled, columns=df_pivot_clean.columns, index=df_pivot_clean.index)
df_scaled['cluster'] = labels_km
centroids = df_scaled.groupby('cluster').mean()

# 3 panel: slow | medium | fast metrikleri
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

metrics_base = ['mean_apex_speed', 'std_apex_speed', 'mean_braking_dist', 
                'std_braking_dist', 'mean_exit_speed', 'std_exit_speed', 'trail_pct']
metric_labels = ['Apex\nSpeed', 'Apex\nStd', 'Brake\nDist', 
                 'Brake\nStd', 'Exit\nSpeed', 'Exit\nStd', 'Trail\n%']

for i, ct in enumerate(CORNER_ORDER):
    ax = axes[i]
    cols = [f'{ct}_{m}' for m in metrics_base]
    
    x = np.arange(len(metrics_base))
    width = 0.25
    
    for c_id in range(3):
        vals = centroids.loc[c_id, cols].values
        ax.bar(x + c_id * width, vals, width, 
               color=CLUSTER_COLORS[c_id], label=CLUSTER_NAMES[c_id],
               edgecolor='white', linewidth=0.5)
    
    ax.set_xticks(x + width)
    ax.set_xticklabels(metric_labels, fontsize=7)
    ax.set_title(f'{ct.capitalize()} Corners', fontsize=10, 
                 color=CORNER_COLORS[ct], fontweight='bold')
    ax.set_ylim(0, 1.05)
    if i == 0:
        ax.set_ylabel('Min-Max Scaled Value', fontsize=9)

axes[2].legend(loc='upper right', fontsize=8)
fig.suptitle('Cluster Centroid Profilleri (Corner-Type Bazli)', fontsize=11, y=1.02)
fig.tight_layout()
save_fig(fig, 'cluster_centroid_bars_by_corner_type')
plt.show()

## 7. Radar Chart — Cluster Centroid Profilleri (Destekleyici)

**Uyari:** Polygon area ~ variable^2 bias (Duan 2023).
Percentile-normalized, semantik aks gruplama, max 3 overlay.
Sayisal tablo ile birlikte sunulur.

In [ ]:
# --- RADAR CHART (DESTEKLEYICI) ---
def radar_chart(ax, categories, values_dict, title, colors):
    """Tek bir radar subplot cizer."""
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # kapat
    
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(0)
    
    for label, vals in values_dict.items():
        vals_closed = vals.tolist() + [vals.tolist()[0]]
        ax.plot(angles, vals_closed, 'o-', linewidth=1.5, markersize=3,
                label=label, color=colors[label])
        ax.fill(angles, vals_closed, alpha=0.08, color=colors[label])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=10, pad=15)

# Percentile normalization (5th-95th)
def percentile_normalize(series):
    p5, p95 = series.quantile(0.05), series.quantile(0.95)
    if p95 == p5:
        return pd.Series(0.5, index=series.index)
    return ((series - p5) / (p95 - p5)).clip(0, 1)

# Centroidleri percentile-normalize et
centroids_pctl = centroids.copy()
# Butun surucular uzerinden percentile hesapla, sonra centroid'lere uygula
for col in df_pivot_clean.columns:
    p5 = df_scaled[col].quantile(0.05)
    p95 = df_scaled[col].quantile(0.95)
    if p95 != p5:
        centroids_pctl[col] = ((centroids[col] - p5) / (p95 - p5)).clip(0, 1)
    else:
        centroids_pctl[col] = 0.5

# 3 subplot: slow | medium | fast
fig, axes = plt.subplots(1, 3, figsize=(14, 5), subplot_kw=dict(polar=True))

radar_colors = {CLUSTER_NAMES[k]: v for k, v in CLUSTER_COLORS.items()}

for i, ct in enumerate(CORNER_ORDER):
    cols = [f'{ct}_{m}' for m in metrics_base]
    vals_dict = {}
    for c_id in range(3):
        vals_dict[CLUSTER_NAMES[c_id]] = centroids_pctl.loc[c_id, cols].values
    
    radar_chart(axes[i], metric_labels, vals_dict, 
                f'{ct.capitalize()} Corners', radar_colors)

axes[2].legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=8)
fig.suptitle('Cluster Centroid Radar (Percentile-Normalized, Destekleyici)', fontsize=11, y=1.05)
fig.tight_layout()
save_fig(fig, 'radar_cluster_centroids_corner_type')
plt.show()

# Sayisal tablo
print('\n=== Centroid Degerleri (Min-Max Scaled) ===')
for c_id in range(3):
    print(f'\n--- {CLUSTER_NAMES[c_id]} ({(labels_km == c_id).sum()} surucu) ---')
    for ct in CORNER_ORDER:
        cols = [f'{ct}_{m}' for m in metrics_base]
        vals = centroids.loc[c_id, cols]
        print(f'  {ct:>6}: ' + '  '.join(f'{v:.2f}' for v in vals))

## 8. Per-Corner-Type Kumeleme + Cluster Kararliligi

Her corner-type icin ayri k=3 kumeleme. ARI (Hubert & Arabie 1985)
ile cross-corner-type cluster uyumu test edilir.

**Soru:** Suruculer corner-type'a gore cluster degistiriyor mu?

In [ ]:
# --- PER-CORNER-TYPE K-MEANS ---
per_ct_labels = {}
per_ct_validity = {}

for ct in CORNER_ORDER:
    # Crosstrack'tan bu corner-type'i filtrele
    ct_data = df_cross_clean[df_cross_clean['corner_type'] == ct].copy()
    ct_data = ct_data.set_index('driver_id')
    
    # Numerik kolonlar
    num_cols = ['mean_apex_speed', 'std_apex_speed', 'mean_braking_dist',
                'std_braking_dist', 'mean_exit_speed', 'std_exit_speed', 'trail_pct']
    X_ct = ct_data[num_cols].values
    
    # Min-Max scale
    scaler_ct = MinMaxScaler()
    X_ct_scaled = scaler_ct.fit_transform(X_ct)
    
    # K-Means k=3
    km_ct = KMeans(n_clusters=3, random_state=42, n_init=20)
    labels_ct = km_ct.fit_predict(X_ct_scaled)
    
    sil_ct = silhouette_score(X_ct_scaled, labels_ct)
    ch_ct  = calinski_harabasz_score(X_ct_scaled, labels_ct)
    db_ct  = davies_bouldin_score(X_ct_scaled, labels_ct)
    
    per_ct_labels[ct] = pd.Series(labels_ct, index=ct_data.index)
    per_ct_validity[ct] = {'Silhouette': sil_ct, 'CH': ch_ct, 'DB': db_ct}
    
    print(f'{ct:>6}: Sil={sil_ct:.3f}  CH={ch_ct:.2f}  DB={db_ct:.3f}  '
          f'sizes={np.bincount(labels_ct).tolist()}')

# --- ARI MATRIX ---
print('\n=== Adjusted Rand Index (Cross-Corner-Type) ===')
ari_matrix = pd.DataFrame(index=CORNER_ORDER, columns=CORNER_ORDER, dtype=float)
for ct1 in CORNER_ORDER:
    for ct2 in CORNER_ORDER:
        # Ortak suruculer
        common = per_ct_labels[ct1].index.intersection(per_ct_labels[ct2].index)
        l1 = per_ct_labels[ct1].loc[common].values
        l2 = per_ct_labels[ct2].loc[common].values
        ari_matrix.loc[ct1, ct2] = adjusted_rand_score(l1, l2)

print(ari_matrix.to_string(float_format='{:.3f}'.format))
print()

# Yorum
mean_ari = ari_matrix.values[np.triu_indices(3, k=1)].mean()
print(f'Ortalama off-diagonal ARI: {mean_ari:.3f}')
if mean_ari > 0.6:
    print('>> Kararli: Suruculer corner-type\'lar arasi tutarli kume yapisi gosteriyor')
elif mean_ari > 0.3:
    print('>> Orta: Kismi farkliliklar var, bazi suruculer corner-type\'a gore degisiyor')
else:
    print('>> Degisken: Corner-type surucu kume yapisini onemli olcude degistiriyor')

## 9. Cluster Uyeligi Gecis Tablosu

Hangi suruculer corner-type'a gore cluster degistiriyor?

In [ ]:
# --- GECIS TABLOSU ---
transition = pd.DataFrame(index=per_ct_labels['slow'].index)
for ct in CORNER_ORDER:
    transition[ct] = per_ct_labels[ct]

transition['driver_short'] = [short_id(d) for d in transition.index]
transition['stable'] = transition['slow'] == transition['medium']
transition.loc[transition['stable'], 'stable'] = (
    transition.loc[transition['stable'], 'medium'] == transition.loc[transition['stable'], 'fast']
)

n_stable = transition['stable'].sum()
n_shifting = len(transition) - n_stable
print(f'Kararli suruculer (ayni cluster): {n_stable}/{len(transition)}')
print(f'Degisen suruculer: {n_shifting}/{len(transition)}')
print()

# Tabloyu goster
display_cols = ['driver_short', 'slow', 'medium', 'fast', 'stable']
print(transition[display_cols].sort_values(['stable', 'slow', 'medium', 'fast']).to_string(index=False))

## 10. Sankey — Cluster Akisi (Slow -> Medium -> Fast)

Plotly Sankey ile gorsellestirilir. HTML export.

In [ ]:
# --- SANKEY DIAGRAM ---
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print('plotly bulunamadi — pip install plotly')
    print('Sankey atlaniyor.')

if HAS_PLOTLY:
    # Node'lar: slow_C0, slow_C1, slow_C2, medium_C0, ..., fast_C2
    labels = []
    node_colors = []
    for ct in CORNER_ORDER:
        for c in range(3):
            labels.append(f'{ct.capitalize()} C{c}')
            node_colors.append(CLUSTER_COLORS[c])
    
    # Link'ler: slow->medium, medium->fast
    sources, targets, values, link_colors = [], [], [], []
    
    for stage_idx, (ct_from, ct_to) in enumerate([
        ('slow', 'medium'), ('medium', 'fast')
    ]):
        for c_from in range(3):
            for c_to in range(3):
                mask = (transition[ct_from] == c_from) & (transition[ct_to] == c_to)
                count = mask.sum()
                if count > 0:
                    src_idx = stage_idx * 3 + c_from       # slow: 0-2, medium: 3-5
                    tgt_idx = (stage_idx + 1) * 3 + c_to   # medium: 3-5, fast: 6-8
                    sources.append(src_idx)
                    targets.append(tgt_idx)
                    values.append(count)
                    # Link rengi: kaynak cluster rengi, alpha
                    hex_c = CLUSTER_COLORS[c_from].lstrip('#')
                    r, g, b = int(hex_c[:2], 16), int(hex_c[2:4], 16), int(hex_c[4:], 16)
                    link_colors.append(f'rgba({r},{g},{b},0.4)')
    
    fig_sankey = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=labels, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=link_colors)
    ))
    fig_sankey.update_layout(
        title='Cluster Uyeligi Akisi: Slow -> Medium -> Fast',
        font=dict(family='serif', size=11),
        width=800, height=450,
    )
    
    # HTML export
    sankey_path = FIG_DIR / 'sankey_cluster_flow.html'
    fig_sankey.write_html(str(sankey_path), include_plotlyjs='cdn')
    print(f'Sankey kaydedildi: {sankey_path}')
    fig_sankey.show()

## 11. Trail Braking Delta (Slow vs Fast Corners)

K11 bulgusu: slow %13.1 vs fast %30.4 (2.3x fark).
Cluster bazli bar chart + overall comparison.

In [ ]:
# --- TRAIL BRAKING BAR CHART ---
# Crosstrack verisinden trail_pct
trail_data = df_cross_clean[['driver_id', 'corner_type', 'trail_pct']].copy()

# Cluster atamasini ekle (pivot-based k=3)
trail_data = trail_data.merge(
    cluster_df[['driver_id', 'cluster']], on='driver_id', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Panel (a): Corner-type bazli overall
ax = axes[0]
trail_by_ct = trail_data.groupby('corner_type')['trail_pct'].agg(['mean', 'std'])
trail_by_ct = trail_by_ct.reindex(CORNER_ORDER)
bars = ax.bar(CORNER_ORDER, trail_by_ct['mean'] * 100, 
              yerr=trail_by_ct['std'] * 100,
              color=[CORNER_COLORS[ct] for ct in CORNER_ORDER],
              edgecolor='white', linewidth=0.5, capsize=4)
ax.set_ylabel('Trail Braking (%)', fontsize=9)
ax.set_title('(a) Trail Braking by Corner Type', fontsize=10)
# Deger etiketleri
for bar, val in zip(bars, trail_by_ct['mean'] * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

# Panel (b): Cluster x Corner-type
ax = axes[1]
trail_grouped = trail_data.groupby(['corner_type', 'cluster'])['trail_pct'].mean().unstack()
trail_grouped = trail_grouped.reindex(CORNER_ORDER) * 100

x = np.arange(len(CORNER_ORDER))
width = 0.25
for c_id in range(3):
    if c_id in trail_grouped.columns:
        vals = trail_grouped[c_id].values
        ax.bar(x + c_id * width, vals, width,
               color=CLUSTER_COLORS[c_id], label=CLUSTER_NAMES[c_id],
               edgecolor='white', linewidth=0.5)

ax.set_xticks(x + width)
ax.set_xticklabels([ct.capitalize() for ct in CORNER_ORDER])
ax.set_ylabel('Trail Braking (%)', fontsize=9)
ax.set_title('(b) Trail Braking by Cluster x Corner Type', fontsize=10)
ax.legend(fontsize=8)

fig.tight_layout()
save_fig(fig, 'trail_braking_corner_type_cluster')
plt.show()

# Sayisal ozet
print('\n=== Trail Braking % Ozet ===')
print(trail_grouped.round(1).to_string())

## 12. ARI Heatmap — Cross-Corner-Type Kumeleme Uyumu

In [ ]:
# --- ARI HEATMAP ---
fig, ax = plt.subplots(figsize=(4, 3.5))

ari_vals = ari_matrix.astype(float)
sns.heatmap(ari_vals, annot=True, fmt='.3f', cmap='YlOrRd', 
            vmin=0, vmax=1, ax=ax, linewidths=0.5,
            xticklabels=[ct.capitalize() for ct in CORNER_ORDER],
            yticklabels=[ct.capitalize() for ct in CORNER_ORDER],
            cbar_kws={'label': 'ARI'})
ax.set_title('Adjusted Rand Index\n(Cross-Corner-Type Cluster Uyumu)', fontsize=10)
fig.tight_layout()
save_fig(fig, 'ari_heatmap_corner_type')
plt.show()

## 13. Ozet ve Sonraki Adimlar

Bu notebook corner-type bazli surucu profil analizini gorsellestirir.

**Uretilen figurler:**
1. `clustermap_corner_type_21metrics` — Birincil fingerprint viz
2. `cluster_centroid_bars_by_corner_type` — Cluster karsilastirma
3. `radar_cluster_centroids_corner_type` — Destekleyici radar
4. `trail_braking_corner_type_cluster` — Trail braking analizi
5. `ari_heatmap_corner_type` — Cross-corner cluster kararliligi
6. `sankey_cluster_flow.html` — Interactive Plotly

**Sonraki:** T2-T4 verileriyle ayni analizi tekrarla (tier karsilastirma).

In [ ]:
# Uretilen dosyalari listele
print('=== Uretilen Figurler ===')
for f in sorted(FIG_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:50s} {size_kb:8.1f} KB')

## 14. Izlenebilirlik kaydi (v2) — K1 mansetlerinin tek kaynagi


In [ ]:
# v2: ARI / silhouette / kume boyutlari / kararli surucu — hepsi JSON'a
import json as _json
from datetime import datetime as _dt
_sizes = cluster_df['cluster'].value_counts().sort_index().tolist()
_ari = {f'{c1}|{c2}': float(ari_matrix.loc[c1, c2])
        for i, c1 in enumerate(CORNER_ORDER) for c2 in CORNER_ORDER[i+1:]}
_info = {
    'notebook': 'NB7A7b_corner_type_viz_v2',
    'run_timestamp': _dt.now().isoformat(timespec='seconds'),
    'identity_policy': POLICY,
    'ablock_excluded': (POLICY == 'P_EXC'),
    'exclude_ids': EXCLUDE_IDS,
    'corner_dir': str(CORNER_DIR),
    'n_pivot_input': int(len(df_pivot)),
    'behavioral_degenerate': list(degenerate_drivers),
    'n_analysis': int(len(df_pivot_clean)),
    'driver_ids_analysis': sorted(str(d) for d in df_pivot_clean.index),
    'pool_k3': {'silhouette': float(sil), 'calinski_harabasz': float(ch),
                'davies_bouldin': float(db), 'cluster_sizes': [int(x) for x in _sizes]},
    'per_corner_type': {ct: {k: float(v) for k, v in d.items()} for ct, d in per_ct_validity.items()},
    'ari_pairs': _ari,
    'ari_mean_offdiag': float(mean_ari),
    'n_stable': int(n_stable),
    'n_shifting': int(n_shifting),
    'n_transition_total': int(len(transition)),
    'stable_driver_ids': sorted(str(d) for d in transition.index[transition['stable']]),
}
_out = CORNER_DIR / 'nb7a7b_info.json'
with open(_out, 'w', encoding='utf-8') as _f:
    _json.dump(_info, _f, indent=2, ensure_ascii=False)
print(f'Izlenebilirlik yazildi: {_out}')
print(_json.dumps(_info, indent=1, ensure_ascii=False)[:1100])
